# IPL Performance Analytics — Data Validation

## 1. Validation objective

This notebook validates the serialized Phase 4 cleaned datasets for schema conformance, row preservation, keys, referential integrity, cricket-aware logical consistency, documented cleaning reconciliation, and raw-file immutability. It is read-only: no dataset is repaired or rewritten here.

## 2. Imports and reproducible loading

Project paths are derived from repository structure. `match_date` is parsed by the cleaned-data loader because CSV does not retain an in-memory datetime dtype.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_clean_data, load_raw_data
from src.validation import validate_project

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 100)
matches, deliveries = load_clean_data(PROJECT_ROOT)
matches_raw, deliveries_raw = load_raw_data(PROJECT_ROOT)
validation_results = validate_project(PROJECT_ROOT)
display(pd.DataFrame({"rows": [len(matches), len(deliveries)], "columns": [matches.shape[1], deliveries.shape[1]]}, index=["matches_clean.csv", "deliveries_clean.csv"]))

,rows,columns
matches_clean.csv,1095,32
deliveries_clean.csv,260920,21


## 3. PASS / WARN / FAIL interpretation

**PASS** means the expected invariant holds. **WARN** records an explainable source limitation that does not invalidate keys or measures. **FAIL** means the cleaned data is not ready for downstream use.

In [2]:
status_summary = validation_results["status"].value_counts().reindex(["PASS", "WARN", "FAIL"], fill_value=0).rename_axis("status").to_frame("checks")
category_summary = validation_results.pivot_table(index="category", columns="status", values="check", aggfunc="count", fill_value=0).reindex(columns=["PASS", "WARN", "FAIL"], fill_value=0)
display(status_summary)
display(category_summary)

,checks
status,
PASS,77
WARN,2
FAIL,0


status,PASS,WARN,FAIL
category,,,
Cleaning reconciliation,4,0,0
Cross-table consistency,3,0,0
Data loss,3,0,0
Dates and seasons,7,0,0
Legal deliveries,8,0,0
Match outcomes,7,0,0
Primary keys,4,0,0
Raw integrity,2,0,0
Referential integrity,2,0,0


## 4. Schema validation

The contract requires exactly 32 match columns and 21 delivery columns, integer-like identifiers and run fields, a parsed date, boolean legal-delivery flag, numeric nullable match measures, valid categorical domains, and no accidental blank strings.

In [3]:
display(validation_results.loc[validation_results["category"].eq("Schema")])
display(matches.dtypes.astype(str).to_frame("match_dtype"))
display(deliveries.dtypes.astype(str).to_frame("delivery_dtype"))

,category,check,expected,actual,status,explanation
0,Schema,Match columns match contract,0 missing; 0 unexpected,0 missing; 0 unexpected,PASS,Exact Phase 4 match schema is required.
1,Schema,Delivery columns match contract,0 missing; 0 unexpected,0 missing; 0 unexpected,PASS,Exact Phase 4 delivery schema is required.
2,Schema,Match identifier/calendar fields are integer-like,all integer-like,all integer-like,PASS,CSV-compatible integer inference is sufficient.
3,Schema,Delivery identifier/run fields are integer-like,all integer-like,all integer-like,PASS,"Includes match, innings position, runs, wicket, and legal-ball fields."
4,Schema,Nullable match measures are numeric,all numeric,all numeric,PASS,Float inference is expected because CSV nulls coexist with numeric values.
5,Schema,match_date is parsed datetime,datetime-like,datetime64[us],PASS,The loader restores the serialized ISO date.
6,Schema,is_legal_delivery is boolean,bool,bool,PASS,CSV values infer cleanly as boolean.
7,Schema,No accidental blank strings,0,0,PASS,Nulls are allowed where documented; blank/whitespace-only strings are not.
8,Schema,Categorical domains contain documented values,0,0,PASS,"Checks match type, result, toss, super over, method, extras, and dismissals."


,match_dtype
match_id,int64
season_raw,str
season_standard,str
season_start_year,int64
date_raw,str
match_date,datetime64[us]
match_year,int64
match_month,int64
match_day,int64
match_type,str


,delivery_dtype
match_id,int64
inning,int64
batting_team_raw,str
batting_team_standard,str
bowling_team_raw,str
bowling_team_standard,str
over,int64
ball,int64
batter,str
bowler,str


## 5. Row counts and primary keys

Rows must match Phase 4 exactly. `matches_clean.match_id` is the primary key; recorded deliveries use `match_id + inning + over + ball` as their uniqueness key.

In [4]:
display(validation_results.loc[validation_results["category"].isin(["Row counts", "Primary keys"])])

,category,check,expected,actual,status,explanation
9,Row counts,Match rows preserved,1095,1095,PASS,No match should be added or removed.
10,Row counts,Delivery rows preserved,260920,260920,PASS,One row remains one recorded delivery.
11,Primary keys,Match IDs are non-null,0,0,PASS,Cleaned primary key.
12,Primary keys,Match IDs are unique,0,0,PASS,One record per match.
13,Primary keys,Delivery match IDs are non-null,0,0,PASS,Required foreign key.
14,Primary keys,Delivery composite key is unique,0,0,PASS,Validated at recorded-delivery grain.


## 6. Referential and cross-table integrity

Every delivery must map to one match, every match must contain deliveries, and delivery batting/bowling teams must equal the two standardized participants for that match.

In [5]:
display(validation_results.loc[validation_results["category"].isin(["Referential integrity", "Cross-table consistency"])])

,category,check,expected,actual,status,explanation
15,Referential integrity,Orphan delivery match IDs,0,0,PASS,0 delivery rows reference orphan IDs.
16,Referential integrity,Matches without deliveries,0,0,PASS,Every cleaned match must have delivery data.
17,Cross-table consistency,Delivery batting team belongs to match,0,0,PASS,Uses standardized match participants.
18,Cross-table consistency,Delivery bowling team belongs to match,0,0,PASS,Uses standardized match participants.
19,Cross-table consistency,Per-match team sets agree,0,0,PASS,No unexpected team appears in a match's deliveries.


## 7. Team consistency

This verifies participant logic and the same four documented mappings across all six match/delivery team roles. Old names may remain only in `_raw` audit fields.

In [6]:
display(validation_results.loc[validation_results["category"].eq("Teams")])

,category,check,expected,actual,status,explanation
20,Teams,Standardized team domain has expected size,15,15,PASS,Four documented consolidations reduce 19 raw labels to 15.
21,Teams,Old team labels absent from standard fields,0,0,PASS,Raw audit fields intentionally retain original labels.
22,Teams,Match participants differ,0,0,PASS,A team cannot play itself.
23,Teams,Delivery batting and bowling teams differ,0,0,PASS,Opposing teams are required on every delivery.
24,Teams,Toss winner is a participant,0,0,PASS,Checked after standardization.
25,Teams,Winner is a participant when present,0,0,PASS,No-result null winners are allowed.
26,Teams,Four documented mappings are applied consistently,0,0,PASS,Checks all six mapped match/delivery fields.


## 8. Venue and city validation

Only Phase 4 mappings are accepted. The Dr DY Patil standard venue retains both Mumbai and Navi Mumbai source city labels and is reported as WARN rather than overwritten.

In [7]:
display(validation_results.loc[validation_results["category"].eq("Venue and city")])
multi_city = matches.groupby("venue_standard")["city_standard"].agg(lambda values: sorted(values.dropna().unique())).loc[lambda values: values.map(len).gt(1)]
display(multi_city.to_frame("observed_standard_cities"))

,category,check,expected,actual,status,explanation
27,Venue and city,Venue mapping matches documented rules,0,0,PASS,No Phase 5 venue mappings are introduced.
28,Venue and city,Standard venue labels are non-empty,0,0,PASS,Every match retains a venue.
29,Venue and city,Bangalore is absent from standard city,0,0,PASS,Bengaluru is the documented standard value.
30,Venue and city,Non-null city mapping matches documented rule,0,0,PASS,Only Bangalore is renamed.
31,Venue and city,Dubai/Sharjah city fills are correct,Dubai=33; Sharjah=18; 0 errors,Dubai=33; Sharjah=18; errors=0,PASS,Only unambiguous venue-derived fills are accepted.
32,Venue and city,Standard city has no missing values,0,0,PASS,Phase 4 filled exactly 51 supported values.
33,Venue and city,Standard venues associated with multiple city labels,0 venues,Dr DY Patil Sports Academy=2,WARN,Dr DY Patil inherits Mumbai and Navi Mumbai from source rows; no geography was invented or overw...


,observed_standard_cities
venue_standard,
Dr DY Patil Sports Academy,"[Mumbai, Navi Mumbai]"


## 9. Date and season validation

Single-year season labels must equal the match year; split-year labels permit either component year. The original season identity remains text, while `season_start_year` is only an ordering helper.

In [8]:
display(validation_results.loc[validation_results["category"].eq("Dates and seasons")])
display(matches.groupby(["season_raw", "season_standard", "season_start_year"])["match_year"].agg(lambda values: sorted(values.unique())).to_frame("observed_match_years"))

,category,check,expected,actual,status,explanation
34,Dates and seasons,All match dates are non-null,0,0,PASS,Dates were parsed on load.
35,Dates and seasons,Date range matches source coverage,2008-04-18 to 2024-05-26,2008-04-18 to 2024-05-26,PASS,"Observed dataset bounds, not a claim of future coverage."
36,Dates and seasons,Season domain is preserved,"['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '...","['2007/08', '2009', '2009/10', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '...",PASS,Split-year labels remain strings.
37,Dates and seasons,Standard season preserves raw identity,0,0,PASS,No invented season mapping.
38,Dates and seasons,Season start year follows documented definition,0,0,PASS,First four digits only.
39,Dates and seasons,Match year is consistent with season label,0,0,PASS,Single-year labels equal match year; split labels permit start or end year.
40,Dates and seasons,Calendar components match date,0,0,PASS,Year/month/day derived columns reconcile exactly.


,,,observed_match_years
season_raw,season_standard,season_start_year,
2007/08,2007/08,2007,[2008]
2009,2009,2009,[2009]
2009/10,2009/10,2009,[2010]
2011,2011,2011,[2011]
2012,2012,2012,[2012]
2013,2013,2013,[2013]
2014,2014,2014,[2014]
2015,2015,2015,[2015]
2016,2016,2016,[2016]


## 10. Run reconciliation

Per-delivery arithmetic and aggregate raw/clean totals must agree. For 1,071 non-D/L matches with a target and first innings, target runs can meaningfully be checked against first-innings runs plus one. D/L and missing-target rows are excluded from that comparison.

In [9]:
display(validation_results.loc[validation_results["category"].eq("Runs")])
display(pd.DataFrame({"raw_total": [deliveries_raw["batsman_runs"].sum(), deliveries_raw["extra_runs"].sum(), deliveries_raw["total_runs"].sum()], "clean_total": [deliveries["batsman_runs"].sum(), deliveries["extra_runs"].sum(), deliveries["total_runs"].sum()]}, index=["batsman_runs", "extra_runs", "total_runs"]))

,category,check,expected,actual,status,explanation
41,Runs,Run values are non-negative,0,0,PASS,Checks all three run columns.
42,Runs,Delivery run arithmetic reconciles,0,0,PASS,total_runs equals batsman_runs plus extra_runs on every row.
43,Runs,Aggregate run totals equal raw data,0,0,PASS,Cleaning did not alter run measures.
44,Runs,Non-D/L targets equal first-innings runs plus one,"0 mismatches among 1,071 eligible matches","0 mismatches among 1,071 eligible matches",PASS,D/L and missing targets are excluded because they are not directly comparable.


,raw_total,clean_total
batsman_runs,330064,330064
extra_runs,17692,17692
total_runs,347756,347756


## 11. Wicket and fielder validation

Dismissal fields are conditionally required only on wicket rows. Caught and stumped dismissals require a named fielder; caught-and-bowled does not because the bowler is implicit. The 181 run outs without a named fielder are retained as a documented source limitation.

In [10]:
display(validation_results.loc[validation_results["category"].eq("Wickets")])
display(deliveries.groupby("dismissal_kind", dropna=False).agg(rows=("match_id", "size"), fielder_present=("fielder", lambda values: values.notna().sum()), fielder_missing=("fielder", lambda values: values.isna().sum())))

,category,check,expected,actual,status,explanation
45,Wickets,Wicket indicator is binary,"{0, 1}","[0, 1]",PASS,No other flags are present.
46,Wickets,Wicket rows have dismissed player,0,0,PASS,Required conditional field.
47,Wickets,Wicket rows have dismissal kind,0,0,PASS,Required conditional field.
48,Wickets,Dismissed-player rows have wicket flag,0,0,PASS,Reverse conditional check.
49,Wickets,Non-wicket rows keep dismissal fields null,0,0,PASS,Confirms conditional null semantics.
50,Wickets,Caught/stumped rows name a fielder,0,0,PASS,Caught-and-bowled is excluded because the bowler is implicit.
51,Wickets,Populated fielders use applicable dismissal types,0,0,PASS,"Only caught, run out, and stumped rows name fielders."
52,Wickets,Run-out rows without named fielder,0,181,WARN,The source omits a fielder on some run outs; values remain null rather than invented.


,rows,fielder_present,fielder_missing
dismissal_kind,,,
bowled,2212,0,2212
caught,8063,8063,0
caught and bowled,367,0,367
hit wicket,15,0,15
lbw,800,0,800
obstructing the field,3,0,3
retired hurt,15,0,15
retired out,3,0,3
run out,1114,933,181


## 12. Legal-delivery validation

Only wides and no-balls are illegal under the documented Phase 4 rule. Ball labels above six remain valid recorded sequence values and do not determine legality.

In [11]:
display(validation_results.loc[validation_results["category"].eq("Legal deliveries")])
display(pd.Series({"recorded": len(deliveries), "legal": deliveries["is_legal_delivery"].sum(), "illegal": (~deliveries["is_legal_delivery"]).sum(), "wides": deliveries["extras_type"].eq("wides").sum(), "no_balls": deliveries["extras_type"].eq("noballs").sum(), "ball_above_6": deliveries["ball"].gt(6).sum()}, name="rows").to_frame())

,category,check,expected,actual,status,explanation
53,Legal deliveries,Legal flag follows extras rule,0,0,PASS,Only wides and no-balls are illegal.
54,Legal deliveries,legal_ball matches boolean flag,0,0,PASS,Integer indicator is safe for aggregation.
55,Legal deliveries,Legal plus illegal equals total,260920,260920,PASS,Complete classification.
56,Legal deliveries,Legal-delivery count matches Phase 4,251471,251471,PASS,Documented baseline.
57,Legal deliveries,Illegal-delivery count matches Phase 4,9449,9449,PASS,Documented baseline.
58,Legal deliveries,Wides plus no-balls reconcile to illegal,9449,9449,PASS,"The dataset stores one extras type per row, supporting exact reconciliation."
59,Legal deliveries,Other extras remain legal,0,0,PASS,"Byes, leg-byes, and penalties are not misclassified."
60,Legal deliveries,Ball > 6 rows are preserved,9340,9340,PASS,Recorded sequence values are not treated as invalid.


,rows
recorded,260920
legal,251471
illegal,9449
wides,8380
no_balls,1069
ball_above_6,9340


## 13. Match outcome validation

Outcome checks preserve conditional nulls and reconcile ties, no-results, super overs, margins, and D/L labels to the established source baseline.

In [12]:
display(validation_results.loc[validation_results["category"].eq("Match outcomes")])
display(matches["result"].value_counts().to_frame("matches"))

,category,check,expected,actual,status,explanation
61,Match outcomes,Decided matches have winners,0,0,PASS,Ties decided by super over retain a winner in this source.
62,Match outcomes,No-results have null winners,0,0,PASS,Preserves five structural nulls.
63,Match outcomes,Tie and no-result counts match baseline,ties=14; no-results=5,ties=14; no-results=5,PASS,No outcomes changed during cleaning.
64,Match outcomes,Result margin is populated only when meaningful,0,0,PASS,Run/wicket wins have margins; ties/no-results do not.
65,Match outcomes,Super-over values and count are valid,"{N, Y}; Y=14","['N', 'Y']; Y=14",PASS,Matches the Phase 1/4 baseline.
66,Match outcomes,Tie and super-over indicators reconcile,0,0,PASS,All 14 source ties correspond to the 14 super-over indicators.
67,Match outcomes,Method is null or documented D/L,0,0,PASS,D/L is populated on 21 matches.


,matches
result,
wickets,578
runs,498
tie,14
no result,5


## 14. Toss validation

This reconciles the cleaned data with the EDA baseline. The rate is descriptive association and does not imply that winning the toss causes a match win.

In [13]:
display(validation_results.loc[validation_results["category"].eq("Toss")])

,category,check,expected,actual,status,explanation
68,Toss,Toss decisions use documented values,"['bat', 'field']","['bat', 'field']",PASS,No unexpected decision category.
69,Toss,Toss outcome reconciles with EDA baseline,"1,090 decided; 554 wins; 50.8%","1,090 decided; 554 wins; 50.8%",PASS,Association only; no causal claim.


## 15. Cleaning-result reconciliation

The standardized domains must reproduce the Phase 4 counts without introducing any new mappings.

In [14]:
display(validation_results.loc[validation_results["category"].eq("Cleaning reconciliation")])

,category,check,expected,actual,status,explanation
70,Cleaning reconciliation,Team labels reconcile,19 -> 15,19 -> 15,PASS,Four mappings only.
71,Cleaning reconciliation,Venue labels reconcile,58 -> 40,58 -> 40,PASS,Eighteen documented mappings only.
72,Cleaning reconciliation,City labels reconcile,36 -> 35,36 -> 35,PASS,Bangalore/Bengaluru consolidation.
73,Cleaning reconciliation,Missing standard cities reconcile,51 raw -> 0 standard,51 raw -> 0 standard,PASS,Only Dubai and Sharjah fills.


## 16. Raw-versus-cleaned data-loss checks

Identifier sequences, delivery keys, numeric totals, and conditional-null counts are reconciled to detect dropped, duplicated, reordered, or semantically altered records.

In [15]:
display(validation_results.loc[validation_results["category"].eq("Data loss")])

,category,check,expected,actual,status,explanation
74,Data loss,Match ID sequence is preserved,identical,identical,PASS,"Detects dropped, duplicated, or reordered match records."
75,Data loss,Delivery key sequence is preserved,identical,identical,PASS,"Detects dropped, duplicated, or reordered deliveries."
76,Data loss,Conditional null counts are preserved,0,0,PASS,"Excludes city_standard, the only documented fill target."


## 17. Raw-file integrity

SHA-256 hashes are compared with the recorded Phase 4 baselines. The notebook never writes either raw or cleaned data.

In [16]:
display(validation_results.loc[validation_results["category"].eq("Raw integrity")])

,category,check,expected,actual,status,explanation
77,Raw integrity,matches.csv SHA-256 matches baseline,8a0394246d2a76b44526565f2245d8f4feff91babe57ab16676a2c3d67aa0846,8a0394246d2a76b44526565f2245d8f4feff91babe57ab16676a2c3d67aa0846,PASS,Raw data must remain immutable.
78,Raw integrity,deliveries.csv SHA-256 matches baseline,142236ea52950795dab266d4aa392c7645bd1a0db4704f0b8058afd96e24e2c2,142236ea52950795dab266d4aa392c7645bd1a0db4704f0b8058afd96e24e2c2,PASS,Raw data must remain immutable.


## 18. Final validation summary

Zero FAIL results are required for downstream readiness. WARN results remain visible because validation should communicate limitations rather than force a perfect-looking dataset.

In [17]:
display(status_summary)
warnings = validation_results.loc[validation_results["status"].eq("WARN"), ["category", "check", "actual", "explanation"]]
display(warnings)
failures = validation_results.loc[validation_results["status"].eq("FAIL")]
assert failures.empty, failures.to_string(index=False)
print(f"FINAL STATUS: PASS WITH {len(warnings)} DOCUMENTED WARNINGS")

,checks
status,
PASS,77
WARN,2
FAIL,0


,category,check,actual,explanation
33,Venue and city,Standard venues associated with multiple city labels,Dr DY Patil Sports Academy=2,Dr DY Patil inherits Mumbai and Navi Mumbai from source rows; no geography was invented or overw...
52,Wickets,Run-out rows without named fielder,181,The source omits a fielder on some run outs; values remain null rather than invented.


FINAL STATUS: PASS WITH 2 DOCUMENTED WARNINGS


## 19. Known limitations and Phase 6 handoff

- Player identity remains name-based because no stable player master ID exists.
- Historical stadium renames remain conservatively separate.
- Dr DY Patil retains two source city labels.
- Some run outs lack a named fielder; no fielder is invented.
- CSV serialization requires explicit date parsing on load.

The cleaned layer is structurally and logically suitable for the next approved phase. Phase 6 should create documented match, team, batting, and bowling metrics without changing these validated cleaning rules.